# 04. Deep Transfer Learning with MobileNetV2
### Reusable Convolutional Representations for Visual Sorting
**Course:** Applied Machine Learning & Deep Learning (AML-DL)

---
## 1. Motivation & Architecture
MobileNetV2 provides state-of-the-art visual feature hierarchies at a fraction of the computational footprint:
- **Inverted Residuals with Linear Bottlenecks:** Retains non-linear capacity while preserving feature manifolds.
- **Parameters:** ~3.5 Million parameters (~14 MB), making it viable for embedded IoT smart waste bins.
- **Pre-trained Backbone:** Leverages low-level filters (Gabor-like edge detectors, color gradients) pre-trained on ImageNet-1K.


In [1]:
import sys
from pathlib import Path
import json
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v2
from torchvision import transforms
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score, f1_score

REPO_ROOT = Path("..").resolve()
MODELS_DIR = REPO_ROOT / "models"
SPLITS_DIR = REPO_ROOT / "splits"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using compute device: {device}")


Using compute device: mps


## 2. Load Pretrained MobileNetV2 Checkpoint
We load the best validation checkpoint saved by `src/train_dl.py`.


In [2]:
with open(SPLITS_DIR / "classes.json", "r") as f:
    meta = json.load(f)
class_names = meta["class_names"]

model = mobilenet_v2(weights=None)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(model.last_channel, 6)
)
model.load_state_dict(torch.load(MODELS_DIR / "mobilenetv2_best.pth", map_location=device))
model.to(device)
model.eval()

print("MobileNetV2 model architecture loaded successfully.")
print(f"Classifier head: {model.classifier}")


MobileNetV2 model architecture loaded successfully.
Classifier head: Sequential(
  (0): Dropout(p=0.3, inplace=False)
  (1): Linear(in_features=1280, out_features=6, bias=True)
)


## 3. Evaluate MobileNetV2 on Held-Out Test Set


In [4]:
test_df = pd.read_csv(SPLITS_DIR / "test.csv")

dl_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dl_preds = []
dl_confs = []
dl_targets = test_df["label"].values

with torch.no_grad():
    for _, row in test_df.iterrows():
        img_path = REPO_ROOT / row["filepath"]
        with Image.open(img_path) as img:
            t = dl_transform(img.convert("RGB")).unsqueeze(0).to(device)
            logits = model(t)
            probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
            dl_preds.append(np.argmax(probs))
            dl_confs.append(np.max(probs))

dl_preds = np.array(dl_preds)
dl_confs = np.array(dl_confs)

acc = accuracy_score(dl_targets, dl_preds)
macro_f1 = f1_score(dl_targets, dl_preds, average="macro")

print(f"MobileNetV2 Test Accuracy: {acc*100:.2f}%")
print(f"MobileNetV2 Macro F1:      {macro_f1:.4f}")
print("Detailed Classification Report:")
print(classification_report(dl_targets, dl_preds, target_names=class_names))


MobileNetV2 Test Accuracy: 84.39%
MobileNetV2 Macro F1:      0.8268
Detailed Classification Report:
              precision    recall  f1-score   support

   cardboard       0.90      0.89      0.89        61
       glass       0.85      0.80      0.82        75
       metal       0.85      0.93      0.89        61
       paper       0.85      0.84      0.85        89
     plastic       0.83      0.82      0.83        72
       trash       0.67      0.70      0.68        20

    accuracy                           0.84       378
   macro avg       0.82      0.83      0.83       378
weighted avg       0.84      0.84      0.84       378

